# gold layer - business ready analytics

this notebook creates business level tables from the cleaned silver layer.
these tables can be used for dashboards, reporting, billing and consumption analysis.

In [0]:
# importing the spark functions needed for aggregations

from pyspark.sql import functions as f


# switching to the project catalog

spark.sql("""
use catalog smart_meter_project_catalog
""")

DataFrame[]

In [0]:
# creating the gold schema

spark.sql("""
create schema if not exists gold
""")

DataFrame[]

In [0]:
# reading the cleaned silver layer

gold_df = spark.table("silver.smart_meter_readings")

# taking a quick look at the cleaned data

display(gold_df)

print(f"total records : {gold_df.count()}")

gold_df.printSchema()

household_id,meter_id,timestamp,units_consumed,ingestion_time,ingestion_date,missing_units,is_valid,city,house_type,avg_daily_consumption,rolling_avg_units,is_spike,possible_outage,meter_mean,meter_stddev,high_deviation
H001,M001,2026-04-01T00:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,null,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T01:00:00.000Z,0.8,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.41,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T02:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.605,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T03:00:00.000Z,0.96,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.5399999999999999,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T04:00:00.000Z,0.72,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.645,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T05:00:00.000Z,2.38,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.6599999999999999,true,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T06:00:00.000Z,4.82,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.9466666666666667,true,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T07:00:00.000Z,4.16,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,1.5,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T08:00:00.000Z,4.21,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,1.8325,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T09:00:00.000Z,3.87,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,2.0966666666666667,false,false,2.8427111801242226,1.9053857837008945,false


total records : 1400
root
 |-- household_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- missing_units: boolean (nullable = true)
 |-- is_valid: boolean (nullable = true)
 |-- city: string (nullable = true)
 |-- house_type: string (nullable = true)
 |-- avg_daily_consumption: integer (nullable = true)
 |-- rolling_avg_units: double (nullable = true)
 |-- is_spike: boolean (nullable = true)
 |-- possible_outage: boolean (nullable = true)
 |-- meter_mean: double (nullable = true)
 |-- meter_stddev: double (nullable = true)
 |-- high_deviation: boolean (nullable = true)



In [0]:
# extracting only the date from the timestamp

gold_df = (
    gold_df
    .withColumn(
        "reading_date",
        f.to_date("timestamp")
    )
)

# extracting only the date from the timestamp
# extracting month for monthly analysis

gold_df = (
    gold_df
    .withColumn(
        "reading_month",
        f.date_format(
            "timestamp",
            "yyyy-MM"
        )
    )
)


In [0]:
display(
    gold_df.select(
        "timestamp",
        "reading_date",
        "reading_month"
    )
)

timestamp,reading_date,reading_month
2026-04-01T00:00:00.000Z,2026-04-01,2026-04
2026-04-01T01:00:00.000Z,2026-04-01,2026-04
2026-04-01T02:00:00.000Z,2026-04-01,2026-04
2026-04-01T03:00:00.000Z,2026-04-01,2026-04
2026-04-01T04:00:00.000Z,2026-04-01,2026-04
2026-04-01T05:00:00.000Z,2026-04-01,2026-04
2026-04-01T06:00:00.000Z,2026-04-01,2026-04
2026-04-01T07:00:00.000Z,2026-04-01,2026-04
2026-04-01T08:00:00.000Z,2026-04-01,2026-04
2026-04-01T09:00:00.000Z,2026-04-01,2026-04


# Golf daily consumption delta table creation


In [0]:
# calculating the daily electricity consumption

daily_consumption_df = (
    gold_df
    .groupBy("reading_date")
    .agg(
        f.sum("units_consumed").alias("total_units_consumed"),
        f.avg("units_consumed").alias("average_units_consumed"),
        f.countDistinct("household_id").alias("total_households"),
        f.countDistinct("meter_id").alias("total_meters")
    )
    .orderBy("reading_date")
)

In [0]:
# checking the daily summary

display(daily_consumption_df)

reading_date,total_units_consumed,average_units_consumed,total_households,total_meters
2026-04-01,591.8060869565217,2.4658586956521735,10,10
2026-04-02,516.2295652173914,2.150956521739131,10,10
2026-04-03,566.7691304347825,2.3615380434782605,10,10
2026-04-04,548.7695652173915,2.2865398550724643,10,10
2026-04-05,497.7765217391303,2.07406884057971,10,10
2026-04-06,451.7795652173912,2.2588978260869563,10,10


In [0]:
# storing the daily summary in the gold layer

(
    daily_consumption_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold.daily_consumption")
)

In [0]:
# reading the saved daily table

display(
    spark.table("gold.daily_consumption")
)

reading_date,total_units_consumed,average_units_consumed,total_households,total_meters
2026-04-01,591.8060869565217,2.4658586956521735,10,10
2026-04-02,516.2295652173914,2.150956521739131,10,10
2026-04-03,566.7691304347825,2.3615380434782605,10,10
2026-04-04,548.7695652173915,2.2865398550724643,10,10
2026-04-05,497.7765217391303,2.07406884057971,10,10
2026-04-06,451.7795652173912,2.2588978260869563,10,10


# Gold Monthly consumption delta table 


In [0]:
# calculating the monthly electricity consumption

monthly_consumption_df = (
    gold_df
    .groupBy("reading_month")
    .agg(
        f.sum("units_consumed").alias("total_units_consumed"),
        f.avg("units_consumed").alias("average_units_consumed"),
        f.countDistinct("household_id").alias("total_households"),
        f.countDistinct("meter_id").alias("total_meters")
    )
    .orderBy("reading_month")
)

In [0]:
# checking the monthly summary

display(monthly_consumption_df)

reading_month,total_units_consumed,average_units_consumed,total_households,total_meters
2026-04,3173.130434782612,2.2665217391304373,10,10


In [0]:
# storing the monthly summary

(
    monthly_consumption_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold.monthly_consumption")
)

In [0]:
# reading the saved monthly table

display(
    spark.table("gold.monthly_consumption")
)

reading_month,total_units_consumed,average_units_consumed,total_households,total_meters
2026-04,3173.130434782612,2.2665217391304373,10,10


# Gold layer City wise consumption

In [0]:
# calculating electricity consumption for each city

city_consumption_df = (
    gold_df
    .groupBy("city")
    .agg(
        f.sum("units_consumed").alias("total_units_consumed"),
        f.avg("units_consumed").alias("average_units_consumed"),
        f.countDistinct("household_id").alias("total_households"),
        f.countDistinct("meter_id").alias("total_meters")
    )
    .orderBy(f.desc("total_units_consumed"))
)

In [0]:
# checking the city level summary

display(city_consumption_df)

city,total_units_consumed,average_units_consumed,total_households,total_meters
Pune,899.9495652173919,3.214105590062114,2,2
Jaipur,888.9360869565211,2.1165144927536215,3,3
Kolkata,624.1495652173912,2.2291055900621117,2,2
Mumbai,434.96608695652185,3.106900621118013,1,1
Delhi,169.03956521739133,1.2074254658385095,1,1
Hyderabad,156.0895652173913,1.1149254658385093,1,1


In [0]:
# storing the city summary

(
    city_consumption_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold.city_consumption")
)

In [0]:
# reading the saved city table

display(
    spark.table("gold.city_consumption")
)

city,total_units_consumed,average_units_consumed,total_households,total_meters
Pune,899.9495652173919,3.214105590062114,2,2
Jaipur,888.9360869565211,2.1165144927536215,3,3
Kolkata,624.1495652173912,2.2291055900621117,2,2
Mumbai,434.96608695652185,3.106900621118013,1,1
Delhi,169.03956521739133,1.2074254658385095,1,1
Hyderabad,156.0895652173913,1.1149254658385093,1,1


# Gold House type consumption

In [0]:
# calculating electricity consumption based on house type

house_type_consumption_df = (
    gold_df
    .groupBy("house_type")
    .agg(
        f.sum("units_consumed").alias("total_units_consumed"),
        f.avg("units_consumed").alias("average_units_consumed"),
        f.countDistinct("household_id").alias("total_households"),
        f.countDistinct("meter_id").alias("total_meters")
    )
    .orderBy(f.desc("total_units_consumed"))
)

In [0]:
# checking the house type summary

display(house_type_consumption_df)

house_type,total_units_consumed,average_units_consumed,total_households,total_meters
Studio,1194.3956521739137,2.843799171842652,3,3
Independent,1012.6917391304346,1.8083781055900618,4,4
Apartment,702.8265217391302,2.5100947204968933,2,2
Villa,263.2165217391304,1.8801180124223602,1,1


In [0]:
# storing the house type summary

(
    house_type_consumption_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold.house_type_consumption")
)

In [0]:
# reading the saved house type table

display(
    spark.table("gold.house_type_consumption")
)

house_type,total_units_consumed,average_units_consumed,total_households,total_meters
Studio,1194.3956521739137,2.843799171842652,3,3
Independent,1012.6917391304346,1.8083781055900618,4,4
Apartment,702.8265217391302,2.5100947204968933,2,2
Villa,263.2165217391304,1.8801180124223602,1,1


# Anomaly summary table


In [0]:
# creating a summary of all detected anomalies

anomaly_summary_df = (
    gold_df
    .agg(
        f.sum(f.col("is_spike").cast("int")).alias("total_spikes"),
        f.sum(f.col("possible_outage").cast("int")).alias("total_possible_outages"),
        f.sum(f.col("high_deviation").cast("int")).alias("total_high_deviations")
    )
)

In [0]:
# checking the anomaly summary

display(anomaly_summary_df)

total_spikes,total_possible_outages,total_high_deviations
44,29,21


In [0]:
# saving the anomaly summary

(
    anomaly_summary_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold.anomaly_summary")
)

In [0]:
display(
    spark.table("gold.anomaly_summary")
)

total_spikes,total_possible_outages,total_high_deviations
44,29,21


# Billing Summary Table in Gold layer


In [0]:
# preparing monthly billing information for each household

billing_summary_df = (
    gold_df
    .groupBy(
        "household_id",
        "reading_month"
    )
    .agg(
        f.sum("units_consumed").alias("total_units_consumed"),
        f.avg("units_consumed").alias("average_units_consumed"),
        f.count("timestamp").alias("total_readings")
    )
    .orderBy(
        "household_id",
        "reading_month"
    )
)

display(billing_summary_df)

household_id,reading_month,total_units_consumed,average_units_consumed,total_readings
H001,2026-04,397.97956521739115,2.8427111801242226,140
H002,2026-04,149.06304347826094,1.064736024844721,140
H003,2026-04,263.2165217391304,1.8801180124223602,140
H004,2026-04,227.73999999999995,1.6267142857142853,140
H005,2026-04,475.0865217391305,3.3934751552795035,140
H006,2026-04,434.96608695652185,3.106900621118013,140
H007,2026-04,590.3899999999999,4.217071428571428,140
H008,2026-04,169.03956521739133,1.2074254658385095,140
H009,2026-04,309.5595652173911,2.2111397515527935,140
H010,2026-04,156.0895652173913,1.1149254658385093,140


In [0]:
# saving the billing summary

(
    billing_summary_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold.billing_summary")
)

display(
    spark.table("gold.billing_summary")
)

household_id,reading_month,total_units_consumed,average_units_consumed,total_readings
H001,2026-04,397.97956521739115,2.8427111801242226,140
H002,2026-04,149.06304347826094,1.064736024844721,140
H003,2026-04,263.2165217391304,1.8801180124223602,140
H004,2026-04,227.73999999999995,1.6267142857142853,140
H005,2026-04,475.0865217391305,3.3934751552795035,140
H006,2026-04,434.96608695652185,3.106900621118013,140
H007,2026-04,590.3899999999999,4.217071428571428,140
H008,2026-04,169.03956521739133,1.2074254658385095,140
H009,2026-04,309.5595652173911,2.2111397515527935,140
H010,2026-04,156.0895652173913,1.1149254658385093,140


#Forecast Dataset
Note: This is not a machine learning model. It's a forecast-ready dataset, which is perfectly suitable for a Data Engineering project. A Data Scientist could later build a forecasting model using this data.

In [0]:
# preparing historical data for short term forecasting

from pyspark.sql import Window

forecast_window = (
    Window
    .partitionBy("meter_id")
    .orderBy("reading_date")
    .rowsBetween(-6, 0)
)

forecast_df = (
    gold_df
    .groupBy(
        "meter_id",
        "reading_date"
    )
    .agg(
        f.sum("units_consumed").alias("daily_units")
    )
    .withColumn(
        "seven_day_average",
        f.avg("daily_units").over(forecast_window)
    )
)

In [0]:
# checking the forecast dataset

display(forecast_df)

meter_id,reading_date,daily_units,seven_day_average
M001,2026-04-01,68.74,68.74
M001,2026-04-02,69.44,69.09
M001,2026-04-03,65.68304347826088,67.95434782608696
M001,2026-04-04,79.62652173913044,70.87239130434783
M001,2026-04-05,63.95,69.48791304347826
M001,2026-04-06,50.54,66.32992753623189
M002,2026-04-01,29.396521739130442,29.396521739130442
M002,2026-04-02,24.729999999999997,27.06326086956522
M002,2026-04-03,27.820000000000004,27.315507246376814
M002,2026-04-04,23.379999999999995,26.33163043478261


#Forecast delta table


In [0]:
# saving the forecasting dataset

(
    forecast_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold.forecast_data")
)

In [0]:
display(
    spark.table("gold.forecast_data")
)

meter_id,reading_date,daily_units,seven_day_average
M001,2026-04-01,68.74,68.74
M001,2026-04-02,69.44,69.09
M001,2026-04-03,65.68304347826088,67.95434782608696
M001,2026-04-04,79.62652173913044,70.87239130434783
M001,2026-04-05,63.95,69.48791304347826
M001,2026-04-06,50.54,66.32992753623189
M002,2026-04-01,29.396521739130442,29.396521739130442
M002,2026-04-02,24.729999999999997,27.06326086956522
M002,2026-04-03,27.820000000000004,27.315507246376814
M002,2026-04-04,23.379999999999995,26.33163043478261


In [0]:
# removing the old billing table

spark.sql("""
drop table if exists gold.billing_summary
""")

DataFrame[]